# Task 3.5: Transfer Learning - Iteration 3

**Project:** Potato Quality Classification (Fresh vs. Rotten)  
**ILO:** 8.2 C & D  

This notebook applies transfer learning using a pre-trained MobileNetV2 model to classify potatoes as fresh or rotten. The pre-trained convolutional base is used for feature extraction (frozen weights), with a custom classification head trained on our dataset.

- **A. Preprocessing** - Resize to 128x128, normalise to [0, 1], no augmentation
- **B. CNN Architecture** - MobileNetV2 (frozen) + custom Dense head
- **C. Training** - Train only the classification head with early stopping
- **D. Evaluation** - Compare against all previous iterations

**Important:** Data augmentation is NOT applied in this iteration to isolate the effect of transfer learning.

---

**Results so far:**
| Model | Accuracy |
|---|---|
| Random Guess | 50.18% |
| MLP Baseline | 79.03% |
| CNN v1 (no augmentation) | 94.35% |
| CNN v2 (with augmentation) | 93.28% |

---
## 1. Imports & Setup

In [ ]:
!pip install seaborn scikit-learn -q

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, Flatten, GlobalAveragePooling2D, BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2

from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)

---
## 2. Data Loading & Preprocessing

### Design choices
| Setting | Value | Rationale |
|---|---|---|
| Image size | 128 x 128 px | Same as previous iterations for fair comparison |
| Normalisation | / 255 | Standard [0, 1] scaling |
| Augmentation | None | Isolate the effect of transfer learning |
| Pre-trained model | MobileNetV2 | Lightweight, fast, good accuracy on ImageNet |

### Why MobileNetV2?
MobileNetV2 was chosen over heavier models like VGG16 or ResNet50 for several reasons:
- It is designed for efficiency, making it suitable for a low-cost sorting system (under EUR 500 hardware)
- It has fewer parameters than VGG16/ResNet50, reducing the risk of overfitting on our relatively small dataset (3,712 images)
- It achieves competitive accuracy on ImageNet while being much faster to train
- Its depthwise separable convolutions are well-suited for mobile/embedded deployment, which aligns with FreshScan's goal of affordable hardware

In [ ]:
# -- Paths --
BASE_PATH  = 'data/processed'
TRAIN_PATH = os.path.join(BASE_PATH, 'train')
VAL_PATH   = os.path.join(BASE_PATH, 'val')
TEST_PATH  = os.path.join(BASE_PATH, 'test')

IMG_SIZE   = (128, 128)
BATCH_SIZE = 32

# -- Remove .ipynb_checkpoints if present (causes extra class error) --
import shutil
for split in ['train', 'val', 'test']:
    cp = os.path.join(BASE_PATH, split, '.ipynb_checkpoints')
    if os.path.exists(cp):
        shutil.rmtree(cp)
        print(f'Removed: {cp}')

# -- No augmentation: only rescaling --
datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

val_gen = datagen.flow_from_directory(
    VAL_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen = datagen.flow_from_directory(
    TEST_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

CLASS_NAMES = list(train_gen.class_indices.keys())
print('Classes      :', train_gen.class_indices)
print('Train samples:', train_gen.samples)
print('Val samples  :', val_gen.samples)
print('Test samples :', test_gen.samples)

---
## 3. CNN Architecture - cnn_v3 (Transfer Learning)

### Architecture overview
The model consists of two parts:

1. **Feature extractor (frozen):** MobileNetV2 pre-trained on ImageNet, with the top classification layer removed. All layers are frozen so the pre-trained weights are not modified during training. This base extracts general visual features (edges, textures, shapes) that transfer well to potato classification.

2. **Classification head (trainable):** A custom head that learns to classify potatoes based on the extracted features:
   - GlobalAveragePooling2D (reduces spatial dimensions to a single vector)
   - Dense(256) + BatchNorm + Dropout(0.5)
   - Dense(2, softmax) for binary classification

Only the classification head is trained. This is the **feature extraction** approach, recommended when the dataset is small.

In [ ]:
# -- Load pre-trained MobileNetV2 (without top layer) --
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,           # Remove the ImageNet classification head
    input_shape=(128, 128, 3)
)

# -- Freeze all layers in the base model --
base_model.trainable = False

print(f'Base model: {base_model.name}')
print(f'Total layers: {len(base_model.layers)}')
print(f'Trainable layers: {sum(1 for l in base_model.layers if l.trainable)}')
print(f'Frozen layers: {sum(1 for l in base_model.layers if not l.trainable)}')

In [ ]:
# -- Build full model: frozen base + trainable head --
cnn_v3 = Sequential([
    base_model,

    # -- Classification Head --
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(2, activation='softmax')
], name='cnn_v3_transfer')

cnn_v3.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_v3.summary()

---
## 4. Training with Early Stopping

Since the base model is frozen, only the classification head (~66K parameters) is being trained. This makes training very fast compared to training a full CNN from scratch.

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history_v3 = cnn_v3.fit(
    train_gen,
    epochs=50,
    validation_data=val_gen,
    callbacks=[early_stop],
    verbose=1
)

print(f'\nTraining stopped after {len(history_v3.history["loss"])} epoch(s).')

---
## 5. Learning Curves

In [ ]:
def plot_learning_curves(history, title_suffix=''):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['loss'],     label='Train loss')
    axes[0].plot(history.history['val_loss'], label='Val loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'Loss {title_suffix}')
    axes[0].legend()

    axes[1].plot(history.history['accuracy'],     label='Train accuracy')
    axes[1].plot(history.history['val_accuracy'], label='Val accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title(f'Accuracy {title_suffix}')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


plot_learning_curves(history_v3, title_suffix='- CNN v3 (Transfer Learning)')

---
## 6. Evaluation on the Test Set

In [ ]:
# -- Predict on test set --
test_gen.reset()
y_pred_proba = cnn_v3.predict(test_gen, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = test_gen.classes

# -- Metrics --
acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='weighted')
rec  = recall_score(y_true, y_pred, average='weighted')
f1   = f1_score(y_true, y_pred, average='weighted')

print('=' * 45)
print(f'  Accuracy  : {acc  * 100:.2f} %')
print(f'  Precision : {prec * 100:.2f} %')
print(f'  Recall    : {rec  * 100:.2f} %')
print(f'  F1 Score  : {f1   * 100:.2f} %')
print('=' * 45)

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# -- Confusion matrix heatmap --
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES
)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - CNN v3 (Test Set)')
plt.tight_layout()
plt.show()

---
## 7. Comparison: All Iterations

We compare all models to evaluate the impact of transfer learning.

In [ ]:
RANDOM_GUESS_ACC = 50.18
MLP_BASELINE_ACC = 79.03
CNN_V1_ACC       = 94.35   # Iteration 1: no augmentation
CNN_V2_ACC       = 93.28   # Iteration 2: with augmentation
cnn_v3_acc       = acc * 100

models   = ['Random\nGuess', 'MLP\nBaseline', 'CNN v1\n(no aug)', 'CNN v2\n(aug)', 'CNN v3\n(transfer)']
acc_vals = [RANDOM_GUESS_ACC, MLP_BASELINE_ACC, CNN_V1_ACC, CNN_V2_ACC, cnn_v3_acc]
colours  = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(models, acc_vals, color=colours, edgecolor='black', width=0.5)

for bar, val in zip(bars, acc_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{val:.2f}%',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax.set_ylim(0, 110)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Model Comparison - All Iterations', fontsize=13)
plt.tight_layout()
plt.show()

print('\n-- Comparison Table --')
print(f'  Random Guess     : {RANDOM_GUESS_ACC:.2f}%')
print(f'  MLP Baseline     : {MLP_BASELINE_ACC:.2f}%')
print(f'  CNN v1 (no aug)  : {CNN_V1_ACC:.2f}%')
print(f'  CNN v2 (aug)     : {CNN_V2_ACC:.2f}%')
print(f'  CNN v3 (transfer): {cnn_v3_acc:.2f}%')
delta_v1 = cnn_v3_acc - CNN_V1_ACC
print(f'\n  vs CNN v1: {delta_v1:+.2f} percentage points')

## 8. Summary & Conclusions Iteration 3

**What changed:** The custom CNN architecture was replaced with a pre-trained MobileNetV2 base (frozen) plus a custom classification head. No data augmentation was applied, to isolate the effect of transfer learning.

**Why MobileNetV2:** Lightweight architecture suitable for deployment on affordable hardware (FreshScan target: under EUR 500). Fewer parameters than VGG16/ResNet50 reduces overfitting risk on a small dataset.

**Results:**
- CNN v1 (no augmentation): 94.35%
- CNN v2 (with augmentation): 93.28%
- CNN v3 (transfer learning): 95.43%
- Effect vs CNN v1: +1.08 percentage points

**Analysis:** Transfer learning with MobileNetV2 achieved the best accuracy so far at 95.43%, outperforming both the custom CNN without augmentation (94.35%) and with augmentation (93.28%). Training stopped after 16 epochs (best weights from epoch 11), with a best validation accuracy of 94.73%. The pre-trained ImageNet features provided a strong foundation for distinguishing fresh from rotten potatoes, particularly for texture and colour patterns that MobileNetV2 already learned from its original training. The precision for rotten potatoes was especially high (99%), meaning very few fresh potatoes were incorrectly classified as rotten, which is important for minimising waste in a real sorting system.

**Next steps for Iteration 4:** Combine transfer learning with data augmentation to see if both techniques together can further improve performance.